#### Student informations (Max. 2 students) should be written in this cell (Student ID, Student Name).

Student 1: Ali Emre Kaya, 150210097 

This homework requires you to work on "Value Iteration with Bellman Update" and "Q-learning" algorithms on a topographic map problem. Please read the notebook carefully and be sure to answer all the questions. If you have any questions you can e-mail: sentop22@itu.edu.tr 

# HOMEWORK-3 REINFORCEMENT LEARNING ON GRID WORLD

In this assignment, you are tasked with developing reinforcement learning algorithms for the problem you've solved in your first assignment: Navigating the Topographic Map Problem.

In Question 1, you will model a Markov Decision Process using Q-value iterations and utilizing bell iterations. In Question 2 the task is to implement Q-learning. Let's start by reviewing The Topographic Map Problem.

### NAVIGATING THE TOPOGRAPHIC MAP


The scenario of this homework is the same with Homework-1. You are tasked with implementing a search engine to solve the **Topographic Map Navigation** problem. This time, instead of uninformed and informed search methods, you will implement Reinforcement Learning approaches.


#### The Problem

Given an NxN board simulating a topographic map, each cell represents an area on the map. Each cell has a number that represents the cost of moving to it. Note! Some cells are impassable and are called cliffs, and their value will be 0.
The start and end points for the robot are located on the board and will be indicated in the input file you receive. The robot's goal is to find a path from the start point to the end point using the algorithms studied.

#### Movement Rules
The robot can move one cell at a time to one of the eight surrounding cells, as shown in the diagram below. Movement in any direction is forbidden if the cell represents a cliff. Diagonal movement is prohibited if one of the two directions that form it contains a cliff. For example, in the diagram below, black cells represent cliffs. The robot is allowed to move only to one of the five white cells.

An example 3x3 grid is shown below:

| ![title](grid.png) |
|:-------------------:|

The actions are:

1. **R** (Right)
2. **RD** (Diagonal Down-Right)
3. **D** (Down)
4. **LD** (Diagonal Down-Left)
5. **L** (Left)
6. **LU** (Diagonal Up-Left)
7. **U** (Up)
8. **RU** (Diagonal Up-Right)

Output format:

The output will be shown in a cell. It will contain a single line in the following format: 

$ <path> <number of developed nodes (removed from the frontier)> <path cost> $

If **no path is found**, write: no path (without any additional information).

##### The **solution path** (or simply solution) is described by the series of actions required to traverse the path. The actions will be separated by hyphens. For example:

R-RD-D-D-LD describes moving right, down-right, down, down, and down-left (use capital letters).

To ensure consistent output for all students, the following order of node development (removal from the frontier) will be applied:

1. **Order of Generation**: Nodes will be developed in the order they were generated. This means that once a node is added to the frontier, it will be removed and expanded based on its position in the queue.

2. **Tie-Breaking Rule**: For nodes that are generated simultaneously (i.e., those that share the same parent), the order of expansion will be determined based on the operator that generated them. The operators will be prioritized in the following sequence:
   - Right (R)
   - Diagonal Down-Right (RD)
   - Down (D)
   - Diagonal Down-Left (LD)
   - Left (L)
   - Diagonal Up-Left (LU)
   - Up (U)
   - Diagonal Up-Right (RU)

 ## Question 1: Value Iteration with Bellman Updates

In this problem, you are tasked to use **Value Iteration** with **Bellman updates** to solve the navigation problem. The goal is to find the optimal policy (i.e., the best action to take in each state) using the **Markov Decision Process (MDP)** framework. Navigating on a grid world problem can be easily represented as an MDP as described below.

### MDP Formulation of Grid World

1. **States** $S$: Each cell on the grid represents a state.
2. **Actions** $A$: The eight possible movement directions.
3. **Transition function** ($P(s' | s, a)$): The probability of transitioning to state $s'$ after taking action $a$ in state $s$. This will be deterministic in this problem (i.e., the robot always moves to a valid neighboring cell).
4. **Rewards** ($R(s, a)$): The immediate reward the agent receives when performing action $a$ in state $s$. For this problem:

- If the robot attempts to move to a cliff (an impassable cell), the transition is invalid, and the reward is **negative infinity**. This reflects the fact that the robot cannot move into these cells, and the robot needs to avoid them.

- **Valid Cells**: When the robot moves to a valid (non-cliff) cell, the reward is **negative** to encourage the robot to minimize the cost. The negative reward corresponds to the **cost** of the cell, and it guides the robot to prefer cells with lower costs.

  - For example, if the robot moves to a cell with a cost of 3, the reward would be `-3`, indicating a penalty for moving to that cell.
  - If the robot moves to a cell with a cost of 1, the reward would be `-1`, indicating a lower penalty.


### Value Iteration Algorithm

The core idea of **Value Iteration** is to iteratively update the **value function** for each state, representing the expected cumulative reward from that state to the goal. The Bellman equation is the basis of the value iteration algorithm for solving MDPs. 

1. **Initialize**: 
   - Set the value $V(s)$ for all states $s$ to a large negative number, except for the goal state, which is set to 0.

2. **Bellman Update**: 
   - If there are $n$ possible states, then there are $n$ Bellman equations, one for each state. For each state $s$, update the value based on the following Bellman equation:

   $$
   V(s) = \max_{a \in A} \left( R(s, a) + \gamma \sum_{s'} P(s' | s, a) V(s') \right)
   $$

   Where:
   - $ R(s, a) $ is the immediate reward for taking action $a$ in state \(s\).
   - $ \gamma $ is the **discount factor** $(0 \leq \gamma \leq 1 $  that determines the importance of future rewards.
   - $ P(s' | s, a) $ is the probability of transitioning to state $ s'$ after taking action $a$ in state $s$, which is deterministic in this case.
   - $V(s')$ is the current value of the next state.

3. **Convergence**: 
   - Repeat the Bellman updates until the value function \(V(s)\) converges, i.e., the values do not change significantly between iterations.

### Bellman Update Formula in Detail

The Bellman update for each state can be expressed as:

$$
V(s) = \max_{a \in A} \left( R(s, a) + \gamma \sum_{s'} P(s' | s, a) V(s') \right)
$$

Where:
- $R(s, a)$ is the reward received after taking action $a$ from state $s$.
- $\gamma $ is the discount factor that determines the weight of future rewards.
- $ \max_{a \in A} $ chooses the action that maximizes the expected future reward.

Since the transition function $ P(s' | s, a) $ is deterministic, we can simplify the Bellman equation to:

$$
V(s) = \max_{a \in A} \left( R(s, a) + \gamma V(s') \right)
$$

Where $ s' $ is the new state that results from taking action $a$ in state $s$.

### Algorithm Steps

1. **Initialization**: 
   - In the beginning, you should set $ V(s) = 0 $ for all states except the goal state. You may set a large initial value for all other states (e.g., $ -\infty $).

2. **Iteration**:
   - For each state $s$, you are required to compute the value $V(s)$ using the Bellman update.

3. **Termination**:
   - Stop iterating when the values converge, i.e., when the changes in the value function are smaller than a given threshold.


For this question, you will need to:

1. **Implement the value Iteration Algorithm** under **valueiteration.py**
   
2. **Reconstruct the Optimal Path** by following the best actions from the start to the end point, using the Q-values.

3. **Print the Result** in the specified format.

NOTE: No moves are made into cliff cells and no diagonal moves are allowed when cliffs are involved.

##### Reading the input file

The program will read all input from a single file called **input.txt**. 

The first line will contain the starting point.
The second line will contain the ending point.
The third line in the file will contain the size of the board.
After that (based on the size), the board will be written. 

You may read the input by running the following cell:

In [1]:
def read_input_file(file_path):
    instances = []
    
    with open(file_path, 'r') as file:
        while True:
            # Read and skip any blank lines
            start_line = file.readline().strip()
            while start_line == "":  # Skip blank lines
                start_line = file.readline().strip()
                if not start_line:  # If end of file reached
                    return instances

            # Read the ending point
            end_line = file.readline().strip()
            while end_line == "":  # Skip blank lines
                end_line = file.readline().strip()
                if not end_line:  # If end of file reached
                    return instances

            # Read the board size
            size_line = file.readline().strip()
            while size_line == "":  # Skip blank lines
                size_line = file.readline().strip()
                if not size_line:  # If end of file reached
                    return instances
            
            # Convert the lines to appropriate types
            start_point = tuple(map(int, start_line.split()))
            end_point = tuple(map(int, end_line.split()))
            size = int(size_line)

            # Read the grid based on the size
            grid = []
            for _ in range(size):
                grid_line = file.readline().strip()
                grid.append(list(map(int, grid_line.split())))

            # Store the instance
            instances.append({
                'start': start_point,
                'end': end_point,
                'size': size,
                'grid': grid
            })
    
    return instances

# Example usage
file_path = 'input.txt'  # Change this if your file is named differently
instances = read_input_file(file_path)

# Display the instances
for i, instance in enumerate(instances):
    print(f"Instance {i + 1}:")
    print(f"  Start: {instance['start']}")
    print(f"  End: {instance['end']}")
    print(f"  Size: {instance['size']}")
    print("  Grid:")
    for row in instance['grid']:
        print("   ", row)
    print()

Instance 1:
  Start: (0, 0)
  End: (4, 4)
  Size: 5
  Grid:
    [1, 1, 1, 1, 1]
    [1, 0, 1, 0, 0]
    [1, 1, 1, 1, 1]
    [1, 0, 0, 0, 1]
    [1, 1, 1, 1, 1]

Instance 2:
  Start: (0, 0)
  End: (5, 5)
  Size: 6
  Grid:
    [1, 1, 1, 1, 1, 1]
    [1, 0, 0, 0, 0, 1]
    [1, 1, 1, 1, 0, 1]
    [1, 0, 0, 1, 0, 1]
    [1, 1, 1, 1, 1, 1]
    [1, 1, 1, 1, 1, 1]

Instance 3:
  Start: (0, 0)
  End: (6, 6)
  Size: 9
  Grid:
    [1, 1, 1, 1, 1, 1, 1, 1, 1]
    [1, 0, 0, 1, 0, 0, 1, 1, 1]
    [1, 1, 1, 1, 1, 1, 1, 1, 1]
    [1, 0, 0, 0, 1, 0, 0, 0, 1]
    [1, 1, 1, 1, 1, 1, 1, 1, 1]
    [1, 0, 0, 1, 0, 1, 1, 0, 1]
    [1, 1, 1, 1, 1, 1, 1, 1, 1]
    [1, 0, 0, 0, 0, 0, 0, 0, 1]
    [1, 1, 1, 1, 1, 1, 1, 1, 1]

Instance 4:
  Start: (0, 0)
  End: (9, 9)
  Size: 10
  Grid:
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
    [1, 0, 0, 0, 0, 1, 0, 0, 0, 1]
    [1, 1, 1, 1, 1, 1, 1, 1, 0, 1]
    [1, 0, 0, 0, 0, 0, 0, 1, 0, 1]
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
    [1, 0, 0, 0, 0, 0, 0, 0, 0, 1]
    [1, 1, 1, 1, 1, 

After implementing value iteration algorithm under **valueiteration.py** as `value_iteration` function, you are expected to call the function and print final Value matrix $V$.

In [2]:
from valueiteration import value_iteration, reconstruct_path, format_solution
import numpy as np

# Run Value Iteration on all instances
for i, instance in enumerate(instances):
    print(f"\n{'='*60}")
    print(f"Solving grid instance {i + 1}")
    print(f"{'='*60}")
    
    grid = instance['grid']
    start = instance['start']
    end = instance['end']
    
    print(f"Start: {start}, End: {end}")
    print(f"Grid size: {instance['size']}x{instance['size']}")
    
    V = value_iteration(grid, start, end)
    
    print("\nFinal Value Function:")
    print(np.round(V, 2))


Solving grid instance 1
Start: (0, 0), End: (4, 4)
Grid size: 5x5

Final Value Function:
[[-5.7  -5.22 -4.69 -5.22 -5.7 ]
 [-5.22  -inf -4.1   -inf  -inf]
 [-4.69 -4.1  -3.44 -2.71 -1.9 ]
 [-4.1   -inf  -inf  -inf -1.  ]
 [-3.44 -2.71 -1.9  -1.    0.  ]]

Solving grid instance 2
Start: (0, 0), End: (5, 5)
Grid size: 6x6

Final Value Function:
[[-6.13 -6.13 -5.7  -5.22 -4.69 -4.1 ]
 [-5.7   -inf  -inf  -inf  -inf -3.44]
 [-5.22 -4.69 -4.1  -3.44  -inf -2.71]
 [-4.69  -inf  -inf -2.71  -inf -1.9 ]
 [-4.1  -3.44 -2.71 -1.9  -1.   -1.  ]
 [-4.1  -3.44 -2.71 -1.9  -1.    0.  ]]

Solving grid instance 3
Start: (0, 0), End: (6, 6)
Grid size: 9x9

Final Value Function:
[[-6.86 -6.51 -6.13 -5.7  -6.13 -6.13 -5.7  -5.7  -5.7 ]
 [-6.51  -inf  -inf -5.22  -inf  -inf -5.7  -5.22 -5.22]
 [-6.13 -5.7  -5.22 -4.69 -4.1  -4.69 -5.22 -5.22 -4.69]
 [-5.7   -inf  -inf  -inf -3.44  -inf  -inf  -inf -4.1 ]
 [-5.22 -4.69 -4.1  -3.44 -2.71 -1.9  -1.9  -2.71 -3.44]
 [-5.22  -inf  -inf -3.44  -inf -1.   -1.   

Construct the solution path by using value matrix in the cell below:

In [3]:
############### TO DO: Construct the solution path ###########################
############### TO DO: Print with the required     ###########################
############### solution format.                   ###########################


from valueiteration import value_iteration, reconstruct_path, format_solution
import numpy as np

print("Solution Paths for All Instances:\n")

for i, instance in enumerate(instances):
    grid = instance['grid']
    start = instance['start']
    end = instance['end']
    
    V = value_iteration(grid, start, end)

    # Printing functions
    path, cost = reconstruct_path(grid, start, end, V)
    solution = format_solution(path, cost)
    
    print(f"Instance {i + 1}: {solution}")
    print("-"*50)

##############################################################################
#                             END OF YOUR CODE                               #
##############################################################################

Solution Paths for All Instances:

Instance 1: R-R-D-D-R-R-D-D 8 8
--------------------------------------------------
Instance 2: D-D-R-R-R-D-D-R-RD 9 9
--------------------------------------------------
Instance 3: R-R-R-D-D-R-D-D-R-RD-D 11 11
--------------------------------------------------
Instance 4: R-R-R-R-R-R-R-R-R-D-D-D-D-D-D-D-D-D 18 18
--------------------------------------------------
Instance 5: D-D-D-D-D-D-D-D-D-D-R-R-R-R-R-R-R-R-R-R-RD 21 21
--------------------------------------------------
Instance 6: D-D-D-D-D-D-D-D-D-D-D-D-R-R-R-R-R-R-R-R-R-R-R-R-D-R-RD-RD 28 28
--------------------------------------------------


## Question 2: Navigating the Topographic Map by Q-learning

In this part of the homework, you will implement Q-learning for solving this navigation problem.

### Q-Learning 

The Q-learning algorithm is a model-free reinforcement learning method that updates the Q-values iteratively based on the agent's interaction with the environment.

For this grid world problem, the Q-value update rule is given as:

$$
Q(s, a) \leftarrow Q(s, a) + \alpha \big[ R(s, a) + \gamma \max_{a'} Q(s', a') - Q(s, a) \big]
$$

#### Where:
- Q(s, a): The current Q-value for state $s$ and action $a$.
- **$\alpha$**: The learning rate ($0 < \alpha \leq 1$). Controls how much the new information overrides the old Q-value.  
- **$R(s, a)$**: The reward received for taking action $a$ in state $s$.  
- **$\gamma$**: The discount factor ($0 \leq \gamma \leq 1$). Determines how much future rewards are considered in the current Q-value update.  
- **$\max_{a'} Q(s', a')$**: The maximum Q-value of the next state $s'$, representing the best possible future reward.  

Q-learning uses **Bellman Equations** and it needs multiple episodes of learning even on the same problem instance. 
#### Main Steps of the Algorithm:
1. **Initialize Q-values**:
   - Start with $Q(s, a) = 0$ (or small random values) for all state-action pairs $ (s, a) $.

2. **Choose an Action**:
   - Use an exploration strategy like $\epsilon$-greedy**:
     - With probability $ \epsilon $, choose a random action (exploration).
     - With probability $ 1-\epsilon $, choose the action $ a $ that maximizes $ Q(s, a) $ (exploitation).

3. **Take Action**:
   - Perform the chosen action $ a $, observe the reward $ R(s, a) $, and move to the next state $ s' $.

4. **Update Q-value**:
   - Update the Q-value of $ (s, a) $ using the formula above.

5. **Repeat**:
   - Continue until the Q-values converge or for a fixed number of episodes.


#### You are expected to implement the Q-value update algorithms within the Qlearning.py file. You should add your code to the designated blank spaces. Feel free to implement or import any required data structure implementations. You may modify this script or create additional scripts as long as you provide comments explaining your implementations.

##### Training and Test Grids

To effectively train your Q-learning algorithm and evaluate its performance, you are provided with separate training and test grids. All training grids are under `qlearningtraining.txt` and test grids are under `qlearningtest.txt`. Your algorithm should train by using all of the training grids this homework includes. You should report your algorithm results by using all of the test grids under qlearningtest.txt

In [5]:
##############################################################################
# Q-Learning Implementation - Training and Testing                     #######
##############################################################################

from qlearning import GridWorld, QLearning, read_qlearning_file
import numpy as np

# Read training and test data
training_instances = read_qlearning_file('qlearningtraining.txt')
test_instances = read_qlearning_file('qlearningtest.txt')

print(f"Loaded {len(training_instances)} training instances")
print(f"Loaded {len(test_instances)} test instances")

# Train on all training instances
print("\n" + "="*60)
print("TRAINING PHASE")
print("="*60)

trained_agents = []
for i, instance in enumerate(training_instances):
    print(f"\nTraining on instance {i + 1}:")
    print(f"  Start: {instance['start']}, Goal: {instance['end']}, Size: {instance['size']}")
    
    env = GridWorld(instance['grid'], instance['start'], instance['end'], instance['obstacles'])
    agent = QLearning(env, alpha=0.1, gamma=0.9, epsilon=0.2)
    rewards = agent.train(episodes=1000, max_steps_per_episode=500)
    
    # Get path using learned policy
    path, reached = agent.get_path()
    if reached:
        actions = '-'.join([a for _, a in path])
        print(f"  Learned path: {actions}")
    else:
        print(f"  Could not find path to goal")
    
    trained_agents.append(agent)

# Test on test instances
print("\n" + "="*60)
print("TESTING PHASE")
print("="*60)

for i, instance in enumerate(test_instances):
    print(f"\nTest instance {i + 1}:")
    print(f"  Start: {instance['start']}, Goal: {instance['end']}, Size: {instance['size']}")
    
    env = GridWorld(instance['grid'], instance['start'], instance['end'], instance['obstacles'])
    agent = QLearning(env, alpha=0.1, gamma=0.9, epsilon=0.2)
    rewards = agent.train(episodes=1000, max_steps_per_episode=500)
    
    # Get path
    path, reached = agent.get_path()
    if reached:
        actions = '-'.join([a for _, a in path])
        print(f"  Path: {actions}")
        print(f"  Path length: {len(path)} steps")
    else:
        print(f"  No path found")
    
    # Print learned policy
    policy = agent.get_policy()
    action_symbols = ['→', '↓', '←', '↑']
    print("  Learned Policy:")
    for x in range(env.rows):
        row_str = "    "
        for y in range(env.cols):
            if instance['grid'][x][y] == 0:
                row_str += 'X '
            elif (x, y) == instance['start']:
                row_str += 'S '
            elif (x, y) == instance['end']:
                row_str += 'G '
            else:
                row_str += action_symbols[policy[x, y]] + ' '
        print(row_str)

##############################################################################
#                             END OF YOUR CODE                               #
##############################################################################

Loaded 7 training instances
Loaded 5 test instances

TRAINING PHASE

Training on instance 1:
  Start: (0, 0), Goal: (4, 4), Size: 5
  Learned path: D-R-D-D-R-R-R-D

Training on instance 2:
  Start: (2, 0), Goal: (4, 4), Size: 5
  Learned path: R-D-R-D-R-R

Training on instance 3:
  Start: (2, 1), Goal: (5, 0), Size: 6
  Learned path: D-D-L-D

Training on instance 4:
  Start: (0, 0), Goal: (7, 7), Size: 8
  Learned path: R-D-R-R-R-D-R-D-D-D-R-R-D-D

Training on instance 5:
  Start: (1, 1), Goal: (4, 4), Size: 8
  Learned path: D-D-D-R-R-R

Training on instance 6:
  Start: (0, 0), Goal: (8, 8), Size: 9
  Learned path: D-R-R-R-D-R-D-D-R-R-R-D-D-R-D-D

Training on instance 7:
  Start: (0, 0), Goal: (5, 5), Size: 6
  Learned path: R-R-D-D-R-R-D-D-D-R

TESTING PHASE

Test instance 1:
  Start: (0, 0), Goal: (4, 2), Size: 6
  Path: D-R-D-R-D-D
  Path length: 6 steps
  Learned Policy:
    X ↓ X ↓ ← ← 
    → ↓ ↓ ↓ ← ← 
    → → ↓ ← ← ← 
    ↑ X ↓ ← X ↑ 
    → → G ← → ↓ 
    ↑ → ↑ ↑ → → 

Test ins